<a href="https://colab.research.google.com/github/senthilp005/barath-portfolio/blob/main/fedtwin_model_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FedTwin — Google Drive-backed iterative FedAvg\n\nReads the labelled OT CSV from `MyDrive/FedTwin/data/` and saves checkpoints, metrics and metadata to `MyDrive/FedTwin/outputs/`. Edit `CSV_PATH` and `TARGET_COLUMN` in the code cell. Choose a Colab GPU runtime before running. The IEEE manuscript's 96.71% value is illustrative, not measured.\n

In [ ]:
# FEDTWIN — Google Drive-backed iterative FedAvg experiment
# Drive stores data, checkpoints and outputs; Colab runs the computation.
# IMPORTANT: 96.71% in the IEEE draft is illustrative, not a measured result.

from google.colab import drive
drive.mount('/content/drive')

import os, copy, json, random
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

ROOT = '/content/drive/MyDrive/FedTwin'
DATA_DIR, OUT_DIR = os.path.join(ROOT, 'data'), os.path.join(ROOT, 'outputs')
CKPT_DIR = os.path.join(OUT_DIR, 'checkpoints')
for p in (DATA_DIR, OUT_DIR, CKPT_DIR): os.makedirs(p, exist_ok=True)

# EDIT these two values to match your Drive file and label column.
CSV_PATH = os.path.join(DATA_DIR, 'your_ot_dataset.csv')
TARGET_COLUMN = 'attack'
SEED, CLIENTS, ROUNDS, LOCAL_EPOCHS = 42, 5, 20, 3
HIDDEN, LR, TEST_SIZE = 64, 0.001, 0.20
MAX_ROWS = None  # optionally cap rows during initial debugging

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__, '| Device:', DEVICE)
if DEVICE.type != 'cuda':
    print('WARNING: GPU inactive. Choose Runtime > Change runtime type > T4 GPU.')

# Load real labelled data directly from Drive.
if not os.path.isfile(CSV_PATH):
    raise FileNotFoundError(f'Place your labelled CSV in {DATA_DIR} and edit CSV_PATH.')
df = pd.read_csv(CSV_PATH)
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target '{TARGET_COLUMN}' missing. Available columns: {df.columns.tolist()}")
df = df.dropna(subset=[TARGET_COLUMN]).copy()
if MAX_ROWS and len(df) > MAX_ROWS:
    df, _ = train_test_split(df, train_size=MAX_ROWS, random_state=SEED,
                             stratify=df[TARGET_COLUMN].astype(str))
y_text = df[TARGET_COLUMN].astype(str)
Xdf = df.drop(columns=[TARGET_COLUMN]).select_dtypes(include=[np.number])
Xdf = Xdf.dropna(axis=1, how='all')
Xdf = Xdf.loc[:, Xdf.nunique(dropna=True) > 1]
if Xdf.shape[1] == 0: raise ValueError('No numeric feature columns found.')
le = LabelEncoder(); y = le.fit_transform(y_text)
if len(le.classes_) < 2: raise ValueError('Need at least two label classes.')
Xtr_df, Xte_df, ytr, yte = train_test_split(
    Xdf, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
imp, scaler = SimpleImputer(strategy='median'), StandardScaler()
Xtr = scaler.fit_transform(imp.fit_transform(Xtr_df)).astype('float32')
Xte = scaler.transform(imp.transform(Xte_df)).astype('float32')
NFEAT, NCLASS = Xtr.shape[1], len(le.classes_)
print('Train/test:', Xtr.shape, Xte.shape, '| Classes:', list(le.classes_))

# kNN graph: nodes are telemetry rows, edges are feature-similar rows.
# This is not an SBOM dependency graph; use real asset/dependency edges for that.
def edge_index_knn(X, k=8):
    k = min(k + 1, len(X))
    ids = NearestNeighbors(n_neighbors=k).fit(X).kneighbors(X, return_distance=False)
    edges = set()
    for i, row in enumerate(ids):
        for j in row:
            if i != int(j): edges.add((i, int(j))); edges.add((int(j), i))
    return torch.tensor(list(edges), dtype=torch.long).t().contiguous()

def graph(X, y=None):
    return (torch.tensor(X, dtype=torch.float32, device=DEVICE),
            edge_index_knn(X).to(DEVICE),
            None if y is None else torch.tensor(y, dtype=torch.long, device=DEVICE))

rng = np.random.default_rng(SEED)
ids = np.arange(len(Xtr)); rng.shuffle(ids)
parts = np.array_split(ids, CLIENTS)
if min(map(len, parts)) < 2: raise ValueError('Not enough rows to create five clients.')
client_graphs = [graph(Xtr[ix], ytr[ix]) for ix in parts]
test_graph = graph(Xte, yte)
print('Client rows:', [len(g[2]) for g in client_graphs])

# Three-layer GraphSAGE mean aggregator, 64 hidden units.
# Implemented in native PyTorch; no CPU-only PyTorch reinstall or PyG compilation.
class SAGE(nn.Module):
    def __init__(self, a, b):
        super().__init__(); self.root = nn.Linear(a,b); self.neigh = nn.Linear(a,b,bias=False)
    def forward(self, x, ei):
        src,dst = ei; agg=torch.zeros_like(x); deg=torch.zeros(x.size(0),device=x.device)
        agg.index_add_(0,dst,x[src]); deg.index_add_(0,dst,torch.ones_like(dst,dtype=x.dtype))
        return self.root(x)+self.neigh(agg/deg.clamp(min=1).unsqueeze(1))

class FedTwinGNN(nn.Module):
    def __init__(self):
        super().__init__(); self.g1=SAGE(NFEAT,HIDDEN); self.g2=SAGE(HIDDEN,HIDDEN)
        self.g3=SAGE(HIDDEN,HIDDEN); self.fc=nn.Linear(HIDDEN,NCLASS)
    def forward(self,x,ei):
        x=F.relu(self.g1(x,ei)); x=F.dropout(x,.3,self.training)
        x=F.relu(self.g2(x,ei)); x=F.dropout(x,.3,self.training)
        return self.fc(F.relu(self.g3(x,ei)))

def new_model(): return FedTwinGNN().to(DEVICE)

def local_fit(global_state, g):
    model=new_model(); model.load_state_dict(global_state); model.train()
    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=1e-4)
    counts=torch.bincount(g[2],minlength=NCLASS).float()
    weights=(counts.sum()/(NCLASS*counts.clamp(min=1))).to(DEVICE)
    loss_fn=nn.CrossEntropyLoss(weight=weights)
    for _ in range(LOCAL_EPOCHS):
        opt.zero_grad(); loss=loss_fn(model(g[0],g[1]),g[2]); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),2.0); opt.step()
    return copy.deepcopy(model.state_dict()),len(g[2]),float(loss.item())

def fedavg(states, sizes):
    total=float(sum(sizes))
    return {k:sum(states[i][k]*(sizes[i]/total) for i in range(len(states)))
            for k in states[0]}

def evaluate(model,g):
    model.eval()
    with torch.no_grad(): pred=model(g[0],g[1]).argmax(1).cpu().numpy()
    truth=g[2].cpu().numpy()
    p,r,f,_=precision_recall_fscore_support(truth,pred,average='weighted',zero_division=0)
    return {'accuracy':accuracy_score(truth,pred),'precision_weighted':p,
            'recall_weighted':r,'f1_weighted':f,'predictions':pred}

# Iterative FedAvg: same global initialization each round + sample-weighted aggregation.
global_model=new_model(); global_state=copy.deepcopy(global_model.state_dict()); history=[]
for rnd in range(1,ROUNDS+1):
    states=[]; sizes=[]; losses=[]
    for g in client_graphs:
        s,n,l=local_fit(global_state,g); states.append(s); sizes.append(n); losses.append(l)
    global_state=fedavg(states,sizes); global_model.load_state_dict(global_state)
    m=evaluate(global_model,test_graph)
    history.append({'round':rnd,'mean_local_loss':float(np.mean(losses)),
                    'test_accuracy':m['accuracy'],'test_weighted_f1':m['f1_weighted']})
    torch.save({'round':rnd,'model_state':global_state},
               os.path.join(CKPT_DIR,f'fedavg_round_{rnd:02d}.pt'))
    print(f"Round {rnd:02d}/{ROUNDS} | loss {np.mean(losses):.4f} | acc {m['accuracy']:.4%} | F1 {m['f1_weighted']:.4%}")

# Centralized comparison with same architecture, train/test split and 60 epochs.
base=new_model(); Xg,Eg,Yg=graph(Xtr,ytr)
opt=torch.optim.Adam(base.parameters(),lr=LR,weight_decay=1e-4)
cnt=torch.bincount(Yg,minlength=NCLASS).float()
cw=(cnt.sum()/(NCLASS*cnt.clamp(min=1))).to(DEVICE)
loss_fn=nn.CrossEntropyLoss(weight=cw); base.train()
for epoch in range(ROUNDS*LOCAL_EPOCHS):
    opt.zero_grad(); loss=loss_fn(base(Xg,Eg),Yg); loss.backward()
    torch.nn.utils.clip_grad_norm_(base.parameters(),2.0); opt.step()

fed=evaluate(global_model,test_graph); central=evaluate(base,test_graph)
results=pd.DataFrame([
 {'approach':'Centralized GNN baseline',**{k:v for k,v in central.items() if k!='predictions'}},
 {'approach':'FedTwin iterative FedAvg',**{k:v for k,v in fed.items() if k!='predictions'}}
])
display(results.style.format({c:'{:.4%}' for c in ['accuracy','precision_weighted','recall_weighted','f1_weighted']}))
print(classification_report(yte,fed['predictions'],target_names=[str(c) for c in le.classes_],zero_division=0))

# Save metrics, models and run metadata to Drive.
stamp=datetime.now().strftime('%Y%m%d_%H%M%S')
pd.DataFrame(history).to_csv(os.path.join(OUT_DIR,f'round_history_{stamp}.csv'),index=False)
results.to_csv(os.path.join(OUT_DIR,f'metrics_{stamp}.csv'),index=False)
torch.save(global_model.state_dict(),os.path.join(OUT_DIR,f'fedtwin_global_{stamp}.pt'))
torch.save(base.state_dict(),os.path.join(OUT_DIR,f'centralized_baseline_{stamp}.pt'))
with open(os.path.join(OUT_DIR,f'metadata_{stamp}.json'),'w') as f:
    json.dump({'device':str(DEVICE),'torch':torch.__version__,'seed':SEED,'clients':CLIENTS,
               'rounds':ROUNDS,'local_epochs':LOCAL_EPOCHS,'hidden':HIDDEN,
               'dataset':CSV_PATH,'target':TARGET_COLUMN,'classes':[str(c) for c in le.classes_],
               'train_rows':len(Xtr),'test_rows':len(Xte)},f,indent=2)
print('Saved checkpoints, metrics and metadata to',OUT_DIR)
print('Measured FedTwin accuracy:',f"{fed['accuracy']:.4%}")
print('IEEE draft 96.71% is illustrative; do not hard-code it as an experimental result.')


Mounted at /content/drive
Torch: 2.11.0+cu130 | Device: cuda


FileNotFoundError: Place your labelled CSV in /content/drive/MyDrive/FedTwin/data and edit CSV_PATH.

In [ ]:

import os

ROOT = "/content/drive/MyDrive/FedTwin"

for folder in [
    os.path.join(ROOT, "data"),
    os.path.join(ROOT, "data", "HAI"),
    os.path.join(ROOT, "data", "HAI", "raw")
]:
    print(f"\nChecking: {folder}")

    if os.path.exists(folder):
        print(os.listdir(folder)[:30])
    else:
        print("Folder does not exist")



Checking: /content/drive/MyDrive/FedTwin/data
['HAI', 'SWaT', 'ToN_IoT']

Checking: /content/drive/MyDrive/FedTwin/data/HAI
['processed', 'raw', 'README.gdoc', 'v2']

Checking: /content/drive/MyDrive/FedTwin/data/HAI/raw
['train1.csv', 'train2.csv', 'train3.csv', 'train4.csv', 'train5.csv', 'train6.csv', 'test1.csv', 'test2.csv', 'test3.csv', 'test4.csv']


In [ ]:

# ================================================================
# FEDTWIN — HAI 22.04 DATA LOADER
# ================================================================

import os
import pandas as pd
import numpy as np

DATA_DIR = "/content/drive/MyDrive/FedTwin/data/HAI/raw"

# Load six normal training files
train_files = [
    os.path.join(DATA_DIR, f"train{i}.csv")
    for i in range(1, 7)
]

# Load four labelled test files
test_files = [
    os.path.join(DATA_DIR, f"test{i}.csv")
    for i in range(1, 5)
]

# Verify files
for path in train_files + test_files:
    if not os.path.isfile(path):
        raise FileNotFoundError(path)

# Read datasets separately
train_dfs = [
    pd.read_csv(path, low_memory=False)
    for path in train_files
]

test_dfs = [
    pd.read_csv(path, low_memory=False)
    for path in test_files
]

print("HAI 22.04 DATASET LOADED")
print("=" * 55)

for i, df in enumerate(train_dfs):
    print(f"Train {i+1}: {df.shape}")

for i, df in enumerate(test_dfs):
    print(f"Test {i+1}: {df.shape}")

print("\nTotal training rows:",
      sum(len(df) for df in train_dfs))

print("Total testing rows:",
      sum(len(df) for df in test_dfs))

print("\nTraining files remain separate for federated clients.")


HAI 22.04 DATASET LOADED
Train 1: (93601, 88)
Train 2: (201600, 88)
Train 3: (126000, 88)
Train 4: (86401, 88)
Train 5: (237600, 88)
Train 6: (259200, 88)
Test 1: (86400, 88)
Test 2: (82800, 88)
Test 3: (62400, 88)
Test 4: (129600, 88)

Total training rows: 1004402
Total testing rows: 361200

Training files remain separate for federated clients.


In [ ]:
# ================================================================
# FEDTWIN V4-G REBUILD | Cell 1: setup, data, model
# ================================================================
import os, random, warnings
import numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.covariance import LedoitWolf
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score)
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

ROOT    = '/content/drive/MyDrive/FedTwin'
RAW_DIR = f'{ROOT}/data/HAI/raw'
V3_CKPT = f'{ROOT}/outputs/V3/fedtwin_v3_best.pt'
V4F_E2  = f'{ROOT}/outputs/V4/V4-F/v4f_no_attention_model.pt'
OUT     = f'{ROOT}/outputs/V4/V4-G-rebuild'
os.makedirs(OUT, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Training budget (same as V4-F E2)
ROUNDS, LOCAL_EPOCHS, LOCAL_BATCH = 20, 1, 512
MAX_WINDOWS, LR, WEIGHT_DECAY, GRAD_CLIP = 12000, 1e-4, 1e-5, 1.0
BATCH_SIZE, DROPOUT = 1024, 0.15

# Scoring and decision rules (same as V4-F)
SENSOR_Q, TOP_K, TOP_K_WEIGHT, MAX_WEIGHT = 0.99, 8, 0.70, 0.30
CURRENT_WEIGHT = 0.40
TEMPORAL_WEIGHTS = {5: 0.25, 10: 0.20, 30: 0.15}
TUNE_FRAC = 0.30
THRESH_QS = np.linspace(0.990, 0.9999, 60)
PERSIST = [(1, 1), (3, 2), (5, 3), (5, 2), (10, 5), (10, 3), (20, 8)]
CLIP = 1000.0
DISCRETE_MAX_UNIQUE = 10

SEEDS = [42, 1, 2, 3]
REUSE_V4F_SEED42 = True   # use the exact saved V4-F E2 weights for seed 42

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

# ---- V3 checkpoint: features, preprocessing, adjacency ----
ckpt = torch.load(V3_CKPT, map_location='cpu', weights_only=False)
features = list(ckpt['feature_columns'])
SEQ = int(ckpt['sequence_length']); HIDDEN = int(ckpt['hidden_size'])
NFEAT = len(features)
imp   = np.asarray(ckpt['imputer_statistics'], dtype=np.float64)
mean  = np.asarray(ckpt['scaler_mean'], dtype=np.float64)
scale = np.asarray(ckpt['scaler_scale'], dtype=np.float64)
scale = np.where(np.abs(scale) < 1e-12, 1.0, scale)
ADJ = torch.tensor(np.asarray(ckpt['adjacency'], dtype=np.float32), device=DEVICE)

def transform(df):
    x = df[features].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    x = x.to_numpy(dtype=np.float64)
    rr, cc = np.where(np.isnan(x))
    if len(rr): x[rr, cc] = imp[cc]
    x = (x - mean) / scale
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def label_array(df):
    names = {'attack', 'label', 'attack_label', 'attacklabel', 'is_attack', 'isattack', 'class'}
    col = next((c for c in df.columns if str(c).strip().lower().replace(' ', '_') in names), None)
    if col is None: raise ValueError(f'No attack label column in {list(df.columns)}')
    v = df[col]; num = pd.to_numeric(v, errors='coerce')
    if num.notna().all(): return (num.to_numpy() > 0).astype(np.int64)
    text = v.astype(str).str.strip().str.lower()
    return (~text.isin({'0', 'normal', 'no_attack', 'no attack', 'false'})).astype(np.int64).to_numpy()

# ---- Data: six normal clients (80/20 chronological), four test files ----
train_frames = [pd.read_csv(f'{RAW_DIR}/train{i}.csv', low_memory=False) for i in range(1, 7)]
test_frames  = [pd.read_csv(f'{RAW_DIR}/test{i}.csv',  low_memory=False) for i in range(1, 5)]
client_train, client_val = [], []
for df in train_frames:
    x = transform(df); cut = int(len(x) * 0.8)
    client_train.append(x[:cut]); client_val.append(x[cut:])
test_arrays = [transform(df) for df in test_frames]
TEST_Y = [label_array(df)[SEQ:] for df in test_frames]

# ---- Discrete columns (training data only) and continuous KEEP index ----
all_train = pd.concat(train_frames, ignore_index=True)
nuniq = all_train[features].nunique(dropna=True)
DISC_IDX = [i for i, c in enumerate(features) if nuniq[c] <= DISCRETE_MAX_UNIQUE]
KEEP = np.array([i for i in range(NFEAT) if i not in set(DISC_IDX)])
print(f'Features: {NFEAT} | excluded discrete: {len(DISC_IDX)} | kept: {len(KEEP)}')

# ---- E2 architecture: no temporal attention ----
class GraphConv(nn.Module):
    def __init__(self, inp, out, graph=True):
        super().__init__(); self.graph = graph; self.linear = nn.Linear(inp, out)
    def forward(self, x, adj=None):
        if self.graph: x = torch.matmul(x, adj)
        return self.linear(x)

class GraphGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.graph1 = GraphConv(NFEAT, HIDDEN, True)
        self.graph2 = GraphConv(HIDDEN, HIDDEN, False)
        self.norm = nn.LayerNorm(HIDDEN); self.activation = nn.GELU()
        self.dropout = nn.Dropout(DROPOUT)
        self.gru = nn.GRU(HIDDEN, HIDDEN, num_layers=1, batch_first=True)
        self.output = nn.Sequential(nn.Linear(HIDDEN, HIDDEN), nn.GELU(),
                                    nn.Dropout(DROPOUT), nn.Linear(HIDDEN, NFEAT))
    def forward(self, x):
        x = self.activation(self.graph1(x, ADJ)); x = self.graph2(x)
        x = self.dropout(self.norm(x))
        seq, _ = self.gru(x)
        return self.output(seq[:, -1, :])

class WindowDS(Dataset):
    def __init__(self, x): self.x = x; self.n = len(x) - SEQ
    def __len__(self): return self.n
    def __getitem__(self, i):
        return torch.from_numpy(self.x[i:i + SEQ]), torch.from_numpy(self.x[i + SEQ])

# Architecture check: V3 weights must load strictly into this class
_m = GraphGRU().to(DEVICE)
_m.load_state_dict(ckpt['model_state_dict'], strict=True)
del _m
print('V3 weights load strictly into the E2 architecture.')